<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.2-data-preparation/notebooks/exercises-7_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.2: Data Preparation for Fine-Tuning

*Module 7 · Fine-Tuning LLMs LIVE CLASS*

> Fine-tuning is only as good as your data. Garbage in = garbage out, but now at scale. This lesson covers the three data formats (instruction, conversation, preference), quality filtering pipelines, and the secret weapon: using LLMs to generate synthetic training data. Every technique coded in Python with reusable functions.

`3 Data Formats` · `Quality Filtering` · `Synthetic Data` · `JSONL Export` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-7.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Three Data Formats — Coded with Examples — `01_data_formats.py`
2. Step 2 — Quality Filtering — Remove Garbage Before Training — `02_quality_filter.py`
3. Step 3 — Synthetic Data Generation — LLMs Creating Training Data — `03_synthetic_data.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Three Data Formats — Coded with Examples

Each format serves a different fine-tuning goal. Know which to use when.

**`01_data_formats.py`** · _Block 1 of 3_

THREE DATA FORMATS FOR FINE-TUNING


In [ ]:
# THREE DATA FORMATS FOR FINE-TUNING
import json

# ── Format 1: INSTRUCTION (OpenAI / Gemini style) ──
instruction_examples = [
    {"messages": [
        {"role": "system", "content": "You are a Netsetos support assistant."},
        {"role": "user", "content": "What is the refund policy?"},
        {"role": "assistant", "content": "Full refund within 7 days. 50% from 8-30 days. None after 30 days."}
    ]},
    {"messages": [
        {"role": "system", "content": "You are a Netsetos support assistant."},
        {"role": "user", "content": "How much does the GenAI course cost?"},
        {"role": "assistant", "content": "The GenAI course costs 14,999 rupees with lifetime access."}
    ]},
]

# ── Format 2: CONVERSATION (multi-turn) ──
conversation_examples = [
    {"messages": [
        {"role": "user", "content": "What courses do you offer?"},
        {"role": "assistant", "content": "We offer GenAI, Data Science, Cloud, and Full-Stack."},
        {"role": "user", "content": "How much is the GenAI one?"},
        {"role": "assistant", "content": "14,999 rupees with lifetime access and GCP credits."},
    ]},
]

# ── Format 3: PREFERENCE / DPO ──
preference_examples = [
    {"prompt": "What is the refund policy?",
     "chosen": "Full refund within 7 days. 50% from 8-30 days. None after 30 days. Contact [email protected].",
     "rejected": "We have a refund policy. You should check the website for details."},
]

# ── Export as JSONL ──
def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"  Saved {len(data)} examples to {path}")

print("Data Formats for Fine-Tuning:\n")

save_jsonl(instruction_examples, "instruction_data.jsonl")
save_jsonl(conversation_examples, "conversation_data.jsonl")
save_jsonl(preference_examples, "preference_data.jsonl")

print(f"\n  Instruction: system + user + assistant (task completion)")
print(f"  Conversation: multi-turn user/assistant pairs (chatbot)")
print(f"  Preference: prompt + chosen + rejected (alignment/DPO)")
print(f"\n  All exported as JSONL (one JSON per line)")


> **What just happened?** Three formats, three purposes: Instruction teaches the model to follow tasks (system + user + assistant). Conversation teaches multi-turn dialogue (4+ alternating messages). Preference/DPO teaches which response is BETTER (chosen vs rejected). All exported as JSONL — the universal format for fine-tuning APIs.


### Step 2 · Quality Filtering — Remove Garbage Before Training

Bad data poisons models. Filter for: length, language, deduplication, and quality scoring.

**`02_quality_filter.py`** · _Block 2 of 3_

QUALITY FILTERING PIPELINE


In [ ]:
# QUALITY FILTERING PIPELINE
import json, re, hashlib
from collections import Counter

class DataFilter:
    """Filter fine-tuning data for quality."""
    def __init__(self, min_response_words=10, max_response_words=500):
        self.min_w, self.max_w = min_response_words, max_response_words
        self.seen_hashes = set()
        self.stats = Counter()

    def _get_response(self, item):
        for m in reversed(item.get("messages",[])):
            if m["role"] == "assistant": return m["content"]
        return ""

    def filter(self, data):
        clean = []
        for item in data:
            self.stats["total"] += 1
            resp = self._get_response(item)

            # Length check
            wc = len(resp.split())
            if wc < self.min_w: self.stats["too_short"] += 1; continue
            if wc > self.max_w: self.stats["too_long"] += 1; continue

            # Deduplication
            h = hashlib.md5(resp.encode()).hexdigest()
            if h in self.seen_hashes: self.stats["duplicate"] += 1; continue
            self.seen_hashes.add(h)

            # Quality heuristics
            if resp.count("I don't know") > 0: self.stats["low_quality"] += 1; continue
            if len(set(resp.lower().split())) / max(len(resp.split()),1) < 0.3:
                self.stats["repetitive"] += 1; continue

            self.stats["kept"] += 1
            clean.append(item)
        return clean

# Demo
raw_data = [
    {"messages": [{"role":"user","content":"Refund?"},{"role":"assistant","content":"Full refund within 7 days of purchase. After 7 days, 50% refund up to 30 days. No refunds after 30 days."}]},
    {"messages": [{"role":"user","content":"Help"},{"role":"assistant","content":"Ok."}]},  # too short
    {"messages": [{"role":"user","content":"Price?"},{"role":"assistant","content":"I don't know the price."}]},  # low quality
    {"messages": [{"role":"user","content":"Refund?"},{"role":"assistant","content":"Full refund within 7 days of purchase. After 7 days, 50% refund up to 30 days. No refunds after 30 days."}]},  # duplicate
    {"messages": [{"role":"user","content":"Cost?"},{"role":"assistant","content":"The GenAI Complete Course costs 14,999 rupees and includes lifetime access to all materials."}]},
]

f = DataFilter(min_response_words=10)
clean = f.filter(raw_data)
print(f"Quality Filtering:\n  {f.stats}\n  {len(raw_data)} raw → {len(clean)} clean")


> **What just happened?** 5 examples filtered: 1 too short (“Ok.”), 1 low quality (“I don’t know”), 1 duplicate (exact same response), 2 passed. Quality filtering typically removes 20-40% of raw data. This is GOOD — 500 high-quality examples outperform 2,000 noisy ones. Data quality > data quantity for fine-tuning.


### Step 3 · Synthetic Data Generation — LLMs Creating Training Data

Don't have enough data? Use a large model to generate training examples for a smaller model.

**`03_synthetic_data.py`** · _Block 3 of 3_

SYNTHETIC DATA GENERATION WITH LLMS


In [ ]:
# SYNTHETIC DATA GENERATION WITH LLMS
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

class SyntheticDataGenerator:
    """Generate fine-tuning data from seed examples + knowledge base."""
    def __init__(self, system_prompt, knowledge_base):
        self.system = system_prompt
        self.kb = knowledge_base
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def generate_instruction(self, n=5):
        resp = self.model.generate_content(
            f"Generate {n} diverse user questions about this knowledge base, "
            f"plus ideal assistant responses.\n"
            f"Format: JSON array of {{\"user\": \"...\", \"assistant\": \"...\"}}\n"
            f"System context: {self.system}\n"
            f"Knowledge:\n{self.kb}\n\nJSON:")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try:
            pairs = json.loads(raw)
            return [{"messages": [
                {"role":"system", "content":self.system},
                {"role":"user", "content":p["user"]},
                {"role":"assistant", "content":p["assistant"]}
            ]} for p in pairs]
        except: return []

    def generate_preference(self, n=3):
        resp = self.model.generate_content(
            f"Generate {n} examples for DPO training.\n"
            f"Format: JSON array of {{\"prompt\":\"...\", \"chosen\":\"...\", \"rejected\":\"...\"}}\n"
            f"'chosen' = helpful, accurate, complete. 'rejected' = vague, wrong, or lazy.\n"
            f"Knowledge:\n{self.kb}\n\nJSON:")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return []

# ── Generate ──
kb = """Netsetos: edtech, Hyderabad. GenAI course: 14999 rupees, 146hrs, 14 modules.
Refund: full 7 days, 50% 8-30, none after 30. Prerequisites: Python, HS math.
Live classes 7PM IST daily. EMI via Razorpay. 5000 students. 85% completion. 4.8 stars."""

gen = SyntheticDataGenerator("You are a helpful Netsetos support assistant.", kb)

print("Synthetic Data Generation:\n")

inst = gen.generate_instruction(n=5)
print(f"  Generated {len(inst)} instruction examples:")
for i in inst[:3]:
    print(f"    Q: {i['messages'][1]['content'][:60]}")
    print(f"    A: {i['messages'][2]['content'][:60]}\n")

pref = gen.generate_preference(n=3)
print(f"  Generated {len(pref)} preference examples:")
for p in pref[:2]:
    print(f"    Prompt: {p.get('prompt','?')[:50]}")
    print(f"    Chosen: {p.get('chosen','?')[:50]}")
    print(f"    Rejected: {p.get('rejected','?')[:50]}\n")


> **What just happened?** The complete pipeline takes seed instructions → production-ready training data in a single automated flow. The quality filtering step (AlpaGasus) is the most impactful — it typically removes 75-85% of data and IMPROVES downstream performance. The pipeline should be versioned and reproducible: every run produces identical results given the same seeds and LLM temperature=0.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
